# Obstacle recognition v2: finish the training on Kaggle (epochs 90-100)

The Colab run stopped at epoch 89 (GPU usage limit). This notebook continues it from `last.pt`, then scores the model on validation and on the held-out test, exactly as the Colab notebook would have.

**Settings** (right panel): Accelerator **GPU T4 x2** (or P100), Internet **On**. **Add Input**: your private dataset `tugger-obstacles` (the three zips: `obstacles_v2.zip`, `obstacles_v1_test.zip`, and the Drive download of the `obstacles_v2` training folder).

Run the cells in order. The last cell gives a link to `obstacle_detector_v2.zip`; it is also under **Output** on the right.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40

In [ ]:
# Find the uploads and rebuild the Colab folder layout (a resumed run expects its old paths).
# Kaggle's input storage is slow per file: copy 32 files at a time, skip files already copied.
import glob, os, shutil, zipfile, yaml, time
from concurrent.futures import ThreadPoolExecutor
IN, TMP = '/kaggle/input', '/kaggle/temp_unzip'
for z in glob.glob(f'{IN}/**/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall(TMP)

def find(test):
    for root in (IN, TMP):
        for d, subdirs, files in os.walk(root):
            if test(d, files):
                return d
    raise FileNotFoundError('not found in the uploads')

data_src = find(lambda d, f: 'data.yaml' in f and os.path.isdir(f'{d}/images/train'))
test_src = find(lambda d, f: os.path.isdir(f'{d}/images/test') and os.path.isdir(f'{d}/labels/test'))
run_src = find(lambda d, f: os.path.isfile(f'{d}/weights/last.pt') and 'results.csv' in f)
print('dataset:', data_src, '\ntest:', test_src, '\ntraining so far:', run_src, flush=True)

def copy_one(pair):
    s, d = pair
    if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s)):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copyfile(s, d)

RUNS = '/content/drive/MyDrive/tugger_runs'
for src, dst in ((run_src, f'{RUNS}/obstacles_v2'), (test_src, '/content/obstacles_v1_test'),
                 (data_src, '/content/obstacles_v2')):
    pairs = [(os.path.join(d, f), os.path.join(dst, os.path.relpath(os.path.join(d, f), src)))
             for d, _, files in os.walk(src) for f in files]
    t = time.time()
    with ThreadPoolExecutor(32) as pool:
        for i, _ in enumerate(pool.map(copy_one, pairs), 1):
            if i % 1000 == 0:
                print(f'  {dst}: {i}/{len(pairs)} files', flush=True)
    print(f'{dst}: {len(pairs)} files in {time.time() - t:.0f} s', flush=True)

cfg = yaml.safe_load(open('/content/obstacles_v2/data.yaml'))
cfg['path'] = '/content/obstacles_v2'
yaml.safe_dump(cfg, open('/content/obstacles_v2/data_colab.yaml', 'w'))
test = dict(cfg, path='/content/obstacles_v1_test', train='images/test', val='images/test')
yaml.safe_dump(test, open('/content/obstacles_v1_test/data_test.yaml', 'w'))
for d in ('/content/obstacles_v2/images/train', '/content/obstacles_v2/images/val',
          '/content/obstacles_v1_test/images/test'):
    print(d, len(os.listdir(d)))          # expect 5052, 948, 1000
!tail -n 1 {RUNS}/obstacles_v2/results.csv | cut -c1-40

In [ ]:
# Continue the training from the last finished epoch (same settings as on Colab).
import numpy as np
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid   # numpy 2 renamed it; Ultralytics 8.3.40 still calls the old name
from ultralytics import YOLO
import ultralytics.utils.callbacks.raytune as raytune
raytune.callbacks.clear()     # Kaggle has a Ray version Ultralytics 8.3.40 cannot talk to; we do not use Ray
RUNS = '/content/drive/MyDrive/tugger_runs'
YOLO(f'{RUNS}/obstacles_v2/weights/last.pt').train(resume=True)

In [ ]:
# Overfitting check: training and validation losses side by side (all 100 epochs).
import pandas as pd, matplotlib.pyplot as plt
RUNS = '/content/drive/MyDrive/tugger_runs'
r = pd.read_csv(f'{RUNS}/obstacles_v2/results.csv')
r.columns = [c.strip() for c in r.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, name in zip(axes, ['box_loss', 'cls_loss']):
    ax.plot(r['epoch'], r[f'train/{name}'], label='train')
    ax.plot(r['epoch'], r[f'val/{name}'], label='validation')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/obstacles_v2/overfitting_check.png', dpi=150); plt.show()

In [ ]:
import numpy as np, json
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid
from ultralytics import YOLO
import ultralytics.utils.callbacks.raytune as raytune
raytune.callbacks.clear()     # Kaggle has a Ray version Ultralytics 8.3.40 cannot talk to; we do not use Ray
RUNS = '/content/drive/MyDrive/tugger_runs'
best = YOLO(f'{RUNS}/obstacles_v2/weights/best.pt')
val = best.val(data='/content/obstacles_v2/data_colab.yaml', imgsz=640, project=RUNS, name='obstacles_v2_val')
test = best.val(data='/content/obstacles_v1_test/data_test.yaml', imgsz=640, project=RUNS, name='obstacles_v2_heldout_test')
summary = {'val_mAP50-95': float(val.box.map), 'val_mAP50': float(val.box.map50),
           'test_mAP50-95': float(test.box.map), 'test_mAP50': float(test.box.map50),
           'test_per_class_mAP50-95': dict(zip(test.names.values(), map(float, test.box.maps)))}
print(summary)
json.dump(summary, open(f'{RUNS}/obstacles_v2/summary.json', 'w'), indent=1)
# The ONNX file is made on the laptop (single file, for OpenCV).

In [ ]:
RUNS = '/content/drive/MyDrive/tugger_runs'
!cd {RUNS} && zip -q -r /kaggle/working/obstacle_detector_v2.zip obstacles_v2 obstacles_v2_val obstacles_v2_heldout_test -x '*/weights/last.pt'
from IPython.display import FileLink
FileLink('obstacle_detector_v2.zip')   # click to download (also under Output on the right)